### Imports + Get as array:

In [2]:
import numpy as np
from pathlib import Path

cell_paths = np.load("metabolite_trajectories.npy", allow_pickle=True)
# trajectories = np.zeros((n_frames, n_metabolites, 3), dtype=np.int32)

# Going to have to reduce by factor of 10. I was told that this was at a 10 Angstrom per voxel, but we do reduce by a factor of 10.
print("Minimum: " + str(np.min(cell_paths)) + " Maximum: " + str(np.max(cell_paths))) 
# Diameter of ~20000 Angstom? syn3A cell is 400 nm diameter... I guess this was 5 Angstrom per voxel?

cell_paths = cell_paths / 10
print(cell_paths)

Minimum: 5.512 Maximum: 2137.5261
[[[ 71.00201  151.37401  136.679   ]
  [ 70.778    151.14801  136.67201 ]
  [ 70.789    150.93701  136.50401 ]
  ...
  [ 44.383003 148.25401   51.408   ]
  [ 69.491005  75.528     99.797005]
  [164.29501   62.758003  53.001   ]]

 [[ 70.791504 151.35152  136.85251 ]
  [ 70.782005 151.22652  136.76102 ]
  [ 70.847    151.11751  136.556   ]
  ...
  [ 44.626503 147.10751   51.192   ]
  [ 69.708     75.878006  99.45601 ]
  [164.85701   64.048004  53.498497]]

 [[ 70.851    151.19334  136.54733 ]
  [ 70.75967  151.09901  136.42233 ]
  [ 70.781    151.02367  136.19868 ]
  ...
  [ 44.77667  146.15167   51.23767 ]
  [ 69.774     75.96267   99.22    ]
  [164.77501   64.334335  53.861004]]

 ...

 [[ 66.4294   183.0458   153.91501 ]
  [ 66.49821  182.92502  153.75821 ]
  [ 66.6182   182.79422  153.6566  ]
  ...
  [ 32.8444   152.68301  206.56961 ]
  [ 51.748005  60.9984    37.664   ]
  [ 13.926801  42.587402  30.6064  ]]

 [[ 66.317604 183.0654   154.08481 ]
  [

In [3]:
# Important files:

# Position of marker
INITIAL_POSITION = [0, 0, 0]
# How far away from the marker we should be
MARKER_DISPLACEMENT = [0, 0, 0]

Print initial array:

In [4]:
files_dir = Path("files")
files_dir.mkdir(exist_ok=True)

n_metabolites = cell_paths.shape[1]
with open(files_dir / "summon.mcfunction", "w") as f:
    for i in range(n_metabolites):
        coord = cell_paths[0][i]

        # summon block_display -956 117 -879 {block_state:{Name:"tnt"},transformation:{scale:[2f,2f,2f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[0f,0f,0f]},Tags:["ce.temp_oxygen","18"],teleport_duration:50}

        f.write(f"summon block_display {coord[0]} {coord[1]} {coord[2]} ")
        f.write("{block_state:{Name:\"purple_concrete\"},transformation:{scale:[1f,1f,1f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[0f,0f,0f]},Tags:[\"ce.moving_martini2\",\"" + str(i) + "\"],teleport_duration:40}")
        f.write("\n")

### Write segments of array into code:

In [5]:
print(cell_paths.shape)
# execute as @n[type=minecraft:block_display,tag=ce.temp_oxygen,tag=18] at @s run data merge entity @s {transformation:{scale:[2f,2f,2f],left_rotation:[0f,1f,0f,0f],right_rotation:[0f,1f,0f,0f],translation:[5f,-37f,-23f]},interpolation_duration:100,start_interpolation:20}


(1121, 84, 3)


In [6]:
smooth = 4

for frame_num in range(cell_paths.shape[0]):
    if frame_num % smooth == 0:
        with open(files_dir / f"tick{frame_num // smooth}.mcfunction", "w") as f:
            for i in range(n_metabolites):
                coord = cell_paths[frame_num][i]

                # execute if entity @s[tag=1] run tp @s 0 0 0
                f.write(f"execute if entity @s[tag={i}] run return run tp @s {coord[0]} {coord[1]} {coord[2]}\n")
